# CNN Parkinson

Ce code entraîne un **CNN (TensorFlow/Keras)** pour classifier des images en 2 classes : `normal` et `parkinson`.

## Chargement des données
La fonction `load_images_and_labels()` parcourt les dossiers du dataset (`parkinsons_dataset/normal` et `parkinsons_dataset/parkinson`), charge les images, les redimensionne en **150x150**, les convertit en tableaux numpy, puis normalise les pixels entre **0 et 1**. Les labels sont créés selon l’index de la classe.

## Split train/test
Les données sont séparées en **80% entraînement** et **20% test** avec `train_test_split`.

## Construction du modèle
La fonction `build_cnn()` construit un modèle `Sequential` avec :
- **Conv2D + ReLU** (extraction des caractéristiques)
- **MaxPooling** (réduction des dimensions)
- **Flatten**
- **Dense(128)** puis **Dense(2, softmax)** pour la classification.

Compilation avec **Adam**, perte **sparse_categorical_crossentropy**, et métrique **accuracy**.

## Entraînement et évaluation
Le modèle est entraîné sur **20 epochs** avec validation sur l’ensemble test.  
Ensuite, on calcule : **accuracy, precision, recall, F1-score** + `classification_report`.

## Sauvegarde
Le modèle est sauvegardé dans `Saved_Models` sous le nom :
`mon_modele_cnn_prk.keras`.

In [1]:
# cnn parkinson
import os
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score, accuracy_score

# Chemin vers le dossier contenant les images
data_directory = "parkinsons_dataset"
image_categories = ['normal', 'parkinson']  # Renommé pour plus de clarté
image_size = (150, 150)

def load_images_and_labels(directory, categories, size):
    images = []
    labels = []
    for category in categories:
        category_index = categories.index(category)
        category_path = os.path.join(directory, category)
        for filename in os.listdir(category_path):
            img_path = os.path.join(category_path, filename)
            img = load_img(img_path, target_size=size)
            img_array = img_to_array(img)
            images.append(img_array)
            labels.append(category_index)
    images = np.array(images, dtype='float32') / 255.0
    labels = np.array(labels)
    return images, labels

all_images, all_labels = load_images_and_labels(data_directory, image_categories, image_size)
train_images, test_images, train_labels, test_labels = train_test_split(all_images, all_labels, test_size=0.2, random_state=42)

def build_cnn(input_shape, num_classes):
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        MaxPooling2D(2, 2),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),
        Conv2D(128, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),
        Flatten(),
        Dense(128, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

model = build_cnn((150, 150, 3), len(image_categories))
model.fit(train_images, train_labels, epochs=20, validation_data=(test_images, test_labels))

# Évaluation du modèle
test_loss, test_accuracy = model.evaluate(test_images, test_labels)
print(f'Test accuracy: {test_accuracy:.2f}, Test loss: {test_loss:.2f}')

# Prédiction des étiquettes de test
y_pred = np.argmax(model.predict(test_images), axis=-1)

# Calcul des mesures de performance
precision = precision_score(test_labels, y_pred, average='weighted')
recall = recall_score(test_labels, y_pred, average='weighted')
f1 = f1_score(test_labels, y_pred, average='weighted')
accuracy = accuracy_score(test_labels, y_pred)

# Affichage des mesures
print(f'Precision: {precision:.2f}')
print(f'Recall: {recall:.2f}')
print(f'F1 Score: {f1:.2f}')
print(f'Accuracy: {accuracy:.2f}')

# Affichage du rapport de classification complet
print(classification_report(test_labels, y_pred, target_names=image_categories))

# Sauvegarde du modèle entraîné
save_path = 'Saved_Models'
if not os.path.exists(save_path):
    os.makedirs(save_path)  # Crée le dossier s'il n'existe pas
model.save(os.path.join(save_path, 'mon_modele_cnn_prk.keras'))

print(f"Modèle sauvegardé sous '{save_path}\\mon_modele_cnn_prk.keras'.")

2025-06-20 18:27:03.530785: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Epoch 1/20


/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


21/21 ━━━━━━━━━━━━━━━━━━━━ 8s 313ms/step - accuracy: 0.7835 - loss: 0.6051 - val_accuracy: 0.8323 - val_loss: 0.2919
Epoch 2/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 7s 309ms/step - accuracy: 0.9012 - loss: 0.2185 - val_accuracy: 0.9760 - val_loss: 0.0889
Epoch 3/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 7s 340ms/step - accuracy: 0.9880 - loss: 0.0627 - val_accuracy: 0.9760 - val_loss: 0.0530
Epoch 4/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 7s 352ms/step - accuracy: 0.9946 - loss: 0.0193 - val_accuracy: 1.0000 - val_loss: 0.0077
Epoch 5/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 10s 332ms/step - accuracy: 0.9925 - loss: 0.0166 - val_accuracy: 0.9820 - val_loss: 0.0285
Epoch 6/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 7s 328ms/step - accuracy: 0.9996 - loss: 0.0110 - val_accuracy: 1.0000 - val_loss: 0.0054
Epoch 7/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 7s 339ms/step - accuracy: 1.0000 - loss: 0.0011 - val_accuracy: 1.0000 - val_loss: 0.0021
Epoch 8/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 7s 333ms/step - accuracy: 1.0000 - loss: 3.2461e-04 - val_accuracy: 1.0000 -